### Import Libraries

In [1]:
# Import warnings
import warnings
warnings.filterwarnings("ignore")




In [1]:
# Import the libraries you will be using for analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os



In [4]:
# Recommended versions
# numpy version: 1.26.4
# pandas version: 2.2.2
# matplotlib version: 3.10.0
# seaborn version: 0.13.2

# Check versions
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)

numpy version: 2.2.6
pandas version: 2.3.3
matplotlib version: 3.10.9
seaborn version: 0.13.2


In [5]:
print(os.getcwd()) #to check the directory

c:\Users\HP\NYC TAXI\Optimising-NYC-Taxi-Operations


### **1.1** Loading the dataset


You will see twelve files, one for each month.

To read parquet files with Pandas, you have to follow a similar syntax as that for CSV files.

`df = pd.read_parquet('file.parquet')`

In [7]:
df = pd.read_parquet('c:\\Users\\HP\\NYC TAXI\\Optimising-NYC-Taxi-Operations\\2025-1.parquet')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

How many rows are there? Do you think handling such a large number of rows is computationally feasible when we have to combine the data for all twelve months into one?

To handle this, we need to sample a fraction of data from each of the files. How to go about that? Think of a way to select only some portion of the data from each month's file that accurately represents the trends.

#### Sampling the Data
> One way is to take a small percentage of entries for pickup in every hour of a date. So, for all the days in a month, we can iterate through the hours and select 5% values randomly from those. Use `tpep_pickup_datetime` for this. Separate date and hour from the datetime values and then for each date, select some fraction of trips for each of the 24 hours.

To sample data, you can use the `sample()` method. Follow this syntax:

```Python
# sampled_data is an empty DF to keep appending sampled data of each hour
# hour_data is the DF of entries for an hour 'X' on a date 'Y'

sample = hour_data.sample(frac = 0.05, random_state = 42)
# sample 0.05 of the hour_data
# random_state is just a seed for sampling, you can define it yourself

sampled_data = pd.concat([sampled_data, sample]) # adding data for this hour to the DF
```

This *sampled_data* will contain 5% values selected at random from each hour.

Note that the code given above is only the part that will be used for sampling and not the complete code required for sampling and combining the data files.

Keep in mind that you sample by date AND hour, not just hour. (Why?)

**1.1.1**
Figure out how to sample and combine the files.

**Note:** It is not mandatory to use the method specified above. While sampling, you only need to make sure that your sampled data represents the overall data of all the months accurately.

In [1]:
# Take a small percentage of entries from each hour of every date.
# Iterating through the monthly data:
#   read a month file -> day -> hour: append sampled data -> move to next hour -> move to next day after 24 hours -> move to next month file
# Create a single dataframe for the year combining all the monthly data

# Select the folder having data files
import os

# Use the current project directory if the trip_records folder is not present.
base_dir = os.getcwd()
trip_records_dir = os.path.join(base_dir, 'trip_records')
if os.path.isdir(trip_records_dir):
    data_dir = trip_records_dir
else:
    data_dir = base_dir

# Create a list of all parquet files to read
file_list = sorted(
    [f for f in os.listdir(data_dir) if f.endswith('.parquet')]
)

# initialise an empty dataframe
final_df = pd.DataFrame()

# iterate through the list of files and sample one by one:
for file_name in file_list:
    try:
        # file path for the current file
        file_path = os.path.join(data_dir, file_name)

        # Reading the current file
        month_df = pd.read_parquet(file_path)
        month_df['tpep_pickup_datetime'] = pd.to_datetime(month_df['tpep_pickup_datetime'])

        # We will store the sampled data for the current date in this df by appending the sampled data from each hour to this
        # After completing iteration through each date, we will append this data to the final dataframe.
        sampled_data = pd.DataFrame()

        # Loop through dates and then loop through every hour of each date
        for pickup_date, day_df in month_df.groupby(month_df['tpep_pickup_datetime'].dt.date):
            daily_sample = pd.DataFrame()

            # Iterate through each hour of the selected date
            for hour in range(24):
                hourly_data = day_df[day_df['tpep_pickup_datetime'].dt.hour == hour]

                if hourly_data.empty:
                    continue

                # Sample 5% of the hourly data randomly
                sample = hourly_data.sample(frac=0.05, random_state=42)

                # add data of this hour to the dataframe
                daily_sample = pd.concat([daily_sample, sample], ignore_index=True)

            sampled_data = pd.concat([sampled_data, daily_sample], ignore_index=True)

        # Concatenate the sampled data of all the dates to a single dataframe
        final_df = pd.concat([final_df, sampled_data], ignore_index=True)
        print(f'Processed {file_name}: {len(sampled_data)} sampled rows')

    except Exception as e:
        print(f'Error reading file {file_name}: {e}')

final_df.head()


SyntaxError: invalid syntax (1507837222.py, line 42)